## Important

**Requirements**

This example requires the optional dependencies for examples. Install them by running:

```bash
pip install scentree[examples]
```

## Utils and tools

This is needed to produce the plots

In [ ]:
# Function to plot data
import matplotlib.pyplot as plt
def plot_figure(
    data,
    target,
    save=False,
    fig_name=None,
    xlabel=None,
):
    plt.figure(figsize=(12, 6))
    plt.plot(
    range(1, data.shape[1] + 1),
    data.T,
    alpha=0.25,
    marker='o',
)
    plt.plot(
        range(1, target.shape[0] + 1),
        target,
        linestyle='--',
        color='black',
        linewidth=2,
        marker='x',
        label="Target day",
    )
    plt.legend(fontsize=12)
    plt.xticks(range(1, Y.shape[1] + 1))
    plt.ylabel("Demand", fontsize=16)
    if xlabel:
        plt.xlabel("Demand period", fontsize=16)
    if save:
        if not fig_name:
            raise ValueError(
                "`fig_name` must be provided"
            )
        plt.savefig(
            fig_name + ".pdf",
            bbox_inches="tight"
        )
    plt.show()

## Load flowergirl data

In [ ]:
import numpy as np
from scentree.io.data import load_flowergirl
from scentree.io.loader import Dataset, DatasetsLoader

# Import and load the data
flowergirl_ds = load_flowergirl()
Y, target_day = flowergirl_ds[:-1],flowergirl_ds[-1]

# Convert the data into a scentree dataset
datasets = [
    Dataset(
        name="demand",
        values=Y,
        stage_ids=[1, 2, 3],
        bounds=(0., np.inf),
    )
]

dsl = DatasetsLoader(datasets=datasets)
full_values = dsl.get_full_values()
stage_ids = dsl.get_sorted_stage_ids()
num_vars = dsl.get_num_variables_per_stage()
stage_cols_map = dsl.create_stage_columns_mapping()

In [ ]:
plot_figure(data=Y, target=target_day)

## Scenario Fan

In [ ]:
from scentree.fan_generator import StageManager

num_fans = 1
num_scenarios = 80
stage_manager = StageManager()
scenario_fans = stage_manager.generate_scenario_fans(
    X=full_values,
    num_fans=num_fans,
    num_scenarios=num_scenarios,
    build_in_sample_fans=False,
    seed=204
)
fan = scenario_fans["scenarios"][0]

In [ ]:
plot_figure(data=fan, target=target_day)

## Scenario tree

In [ ]:
from scentree.tree_construction.ftc import FTC

tree_instance = FTC(
    scenarios=scenario_fans["scenarios"],
    num_variables_per_stage=num_vars,
    stage_ids=stage_ids
)
tree_structure = tree_instance.generate_scenario_trees(
    r=1.,
    stage_columns_mapping=stage_cols_map,
    initial_stage_id_to_cluster=1,
    seed=15,
)
tree_nac = tree_structure["nac_formulation"][0]
tree_data = tree_nac["scenario_tree_data"]
tree_data_matrix = np.unique(tree_data, axis=0)

In [ ]:
plot_figure(data=tree_data_matrix, target=target_day)

## Plot all figures

In [ ]:
# Historical data
plot_figure(data=Y,target=target_day)
# Fan
plot_figure(data=fan, target=target_day)
# Tree
plot_figure(data=tree_data_matrix, target=target_day)

## Optimization problem

In [ ]:
ndl_formulation = tree_structure['nodal_formulation'][0]
parent = ndl_formulation["parent"]
stage = ndl_formulation["stage"]
xi = ndl_formulation["random_variable_value"]
pi = ndl_formulation["probability"]
members = ndl_formulation["node_scenarios"]
nodes = ndl_formulation['nodes']

In [ ]:
import pyomo.environ as pyo
def flowergirl_model(parent, stage, xi, pi):
    root = next(n for n in stage if parent[n] is None)
    N = list(stage)
    B = [n for n in N if n != root]
    O = [n for n in N if stage[n] < 3]
    F = [n for n in N if stage[n] == 3]
    m = pyo.ConcreteModel()
    m.b, m.u, m.ell = 0.85, 1.25, 0.5
    m.x = pyo.Var(O, domain=pyo.NonNegativeReals)
    m.zeta_plus = pyo.Var(
        N, domain=pyo.NonNegativeReals
    )
    m.zeta_minus = pyo.Var(
        B, domain=pyo.NonNegativeReals
    )
    m.zeta_plus[root].fix(0.)
    m.zeta = pyo.Expression(
        B,
        rule=lambda m, n:m.zeta_plus[n] - m.zeta_minus[n]
    )
    def balance(m, n):
        a = parent[n]
        prod = m.x[a] + m.ell * m.zeta_plus[a]
        return prod - xi[n] == m.zeta[n]
    m.balance = pyo.Constraint(B, rule=balance)
    m.profit = pyo.Objective(
        sense=pyo.maximize,
        expr=sum(pi[n] * xi[n] for n in B)
        - m.b * sum(pi[n] * m.x[n] for n in O)
        - m.u * sum(pi[n] * m.zeta_minus[n] for n in B)
        + m.ell * sum(pi[n] * m.zeta_plus[n] for n in F)
    )
    return m

model = flowergirl_model(parent, stage, xi, pi)
solver = pyo.SolverFactory("appsi_highs")
solution = solver.solve(model)

In [ ]:
root = 0
print(f"Initial order: {pyo.value(model.x[root]):.6f}")
print(f"Expected profit on the generated tree: {pyo.value(model.profit):.6f}")

## Generate the graph

In [ ]:
N = list(stage)
B = [n for n in N if n != root]
children = {
    n: [c for c in B if parent[c] == n] for n in N
}
leaves = [n for n in N if not children[n]]

In [ ]:
import pandas as pd
topology = tuple(sum(stage[n] == s for n in N) for s in range(4))
print('Topology:', topology)

# Export the reduced terminal paths and their probabilities.
leaf_rows = []
for leaf in leaves:
    fan_rows = members[leaf]
    leaf_rows.append({
        'leaf_node': leaf,
        'period1': tree_data[fan_rows[0], 0],
        'period2': tree_data[fan_rows[0], 1],
        'period3': tree_data[fan_rows[0], 2],
        'probability': pi[leaf],
        'fan_scenario_ids': ';'.join(str(i+1) for i in fan_rows),
    })
terminal_scenarios = pd.DataFrame(leaf_rows)

O = [n for n in N if stage[n] < 3]
rows = []
for n in N:
    zeta = 0. if n == root else float(pyo.value(model.zeta[n]))
    xval = float(pyo.value(model.x[n])) if n in O else np.nan
    rows.append({
        'node': n,
        'stage': stage[n],
        'parent': parent[n],
        'fan_scenario_ids': ';'.join(str(i+1) for i in members[n]),
        'demand': xi[n],
        'probability': pi[n],
        'x': xval,
        'zeta': zeta,
    })
policy = pd.DataFrame(rows)


In [ ]:
from matplotlib.patches import FancyBboxPatch
policy = pd.DataFrame(rows)
# Arrange terminal scenarios top-down using a depth-first traversal.
def descendant_leaves(n):
    if not children[n]:
        return [n]
    out = []
    for c in children[n]:
        out.extend(descendant_leaves(c))
    return out

leaf_order = descendant_leaves(root)
leaf_omega = {leaf: i+1 for i, leaf in enumerate(leaf_order)}

def omega_ids(n):
    return [leaf_omega[l] for l in descendant_leaves(n)]

def omega_text(n):
    ids = omega_ids(n)
    if len(ids) == 1:
        return rf'$\omega_{{{ids[0]}}}$'
    if len(ids) <= 4:
        return '$' + ','.join(rf'\omega_{{{i}}}' for i in ids) + '$'
    # If contiguous, use the compact reference style.
    if ids == list(range(ids[0], ids[-1]+1)):
        return rf'$\omega_{{{ids[0]}}},\ldots,\omega_{{{ids[-1]}}}$'
    return rf'$|\Omega|={len(ids)}$'

# Layout: root centered, descendants aligned to leaf positions.
xpos = {0:0.10, 1:0.37, 2:0.65, 3:0.91}
leaf_y_values = np.linspace(0.92, 0.08, len(leaf_order))
ypos = {leaf: float(y) for leaf, y in zip(leaf_order, leaf_y_values)}
for s in (2,1,0):
    for n in [k for k in N if stage[k] == s]:
        ypos[n] = float(np.mean([ypos[l] for l in descendant_leaves(n)]))

policy_idx = policy.set_index('node')
def node_lines(n):
    r = policy_idx.loc[n]
    s = int(r.stage)
    lines = [rf'$n={n}$; {omega_text(n)}']
    if s > 0:
        lines.append(rf'$\xi_{{{n}}}={r.demand.item():.3f}$')
    lines.append(rf'$\pi_{{{n}}}={r.probability.item():.3f}$')
    if s < 3:
        lines.append(rf'$x^*({n})={r.x.item():.4f}$')
    lines.append(rf'$\zeta^*({n})={r.zeta.item():.4f}$')
    return lines

fig, ax = plt.subplots(figsize=(8.4, 8.7))
ax.set_xlim(0,1)
ax.set_ylim(-0.10,1.00)
ax.axis('off')

# Edges first.
for n in N:
    p = parent[n]
    if p is None:
        continue
    ax.plot([xpos[stage[p]]+0.065, xpos[stage[n]]-0.065], [ypos[p], ypos[n]],
            lw=0.65, color='black', zorder=1)

# Boxes.
width = {0:0.155, 1:0.150, 2:0.150, 3:0.145}
height = {0:0.071, 1:0.077, 2:0.077, 3:0.064}
fontsize = {0:6.7, 1:6.2, 2:6.2, 3:5.8}
for n in N:
    s = stage[n]
    w, h = width[s], height[s]
    box = FancyBboxPatch(
        (xpos[s]-w/2, ypos[n]-h/2), w, h,
        boxstyle='round,pad=0.004,rounding_size=0.004',
        facecolor='white', edgecolor='black', linewidth=0.70, zorder=2
    )
    ax.add_patch(box)
    ax.text(xpos[s], ypos[n], '\n'.join(node_lines(n)),
            ha='center', va='center', fontsize=fontsize[s], linespacing=0.94, zorder=3)

labels = [
    ('Root', 'Order $x_0$'),
    ('Stage 1', 'Observe $\\xi_1$\nOrder $x_1$'),
    ('Stage 2', 'Observe $\\xi_2$\nOrder $x_2$'),
    ('Stage 3', 'Observe $\\xi_3$\nSettlement'),
]
for s,(a,b) in enumerate(labels):
    ax.text(xpos[s], -0.025, a, ha='center', va='center', fontsize=8.0)
    ax.text(xpos[s], -0.064, b, ha='center', va='center', fontsize=7.1, linespacing=1.0)
plt.show()

